# 04 — Análise Espacial
Mapear a dispersão da contaminação pelos bairros de Goiânia.

Análises:
- Mapa estático dos pontos de contaminação
- Mapa interativo com Folium
- Correlação distância do epicentro × nível de contaminação

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium
from folium.plugins import HeatMap
from math import radians, cos, sin, asin, sqrt

locais = pd.read_csv("../data/processed/locais_clean.csv")
df = pd.read_csv("../data/processed/vitimas_clean.csv")
print(locais[['local', 'lat', 'lon', 'nivel_contaminacao', 'atividade_GBq']])


## 4.1 — Mapa Estático

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
fig.patch.set_facecolor('#F0F4F8')
ax.set_facecolor('#E8F4EA')

cores_nivel = {'critico': '#D32F2F', 'alto': '#F57C00', 'moderado': '#FBC02D', 'baixo': '#388E3C'}
tamanhos = {'critico': 300, 'alto': 200, 'moderado': 130, 'baixo': 80}

for _, row in locais.iterrows():
    nivel = row['nivel_contaminacao']
    ax.scatter(row['lon'], row['lat'], s=tamanhos[nivel], c=cores_nivel[nivel],
               alpha=0.85, edgecolors='white', linewidths=1.5, zorder=5)
    ax.annotate(row['bairro'], (row['lon'], row['lat']),
                textcoords="offset points", xytext=(8, 4),
                fontsize=7.5, fontweight='bold', color='#1A237E',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'))

ponto_zero = locais.iloc[0]
for r in [0.01, 0.02, 0.04]:
    circle = plt.Circle((ponto_zero['lon'], ponto_zero['lat']), r,
                         fill=False, edgecolor='#D32F2F', alpha=max(0.05, 0.3 - r*3), linewidth=1)
    ax.add_patch(circle)

patches = [mpatches.Patch(color=c, label=n.capitalize()) for n, c in cores_nivel.items()]
ax.legend(handles=patches, title="Nivel de Contaminacao", loc='lower left', framealpha=0.9, fontsize=9)
ax.set_xlim(-49.31, -49.19)
ax.set_ylim(-16.74, -16.60)
ax.set_title("Pontos de Contaminacao — Acidente Radiologico de Goiania (1987)", fontsize=12, fontweight='bold')
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/figures/04_mapa_contaminacao.png", dpi=150, bbox_inches='tight')
plt.show()
print("Mapa estatico salvo")


## 4.2 — Mapa Interativo com Folium

In [ ]:
import os
os.makedirs("../reports/dashboard", exist_ok=True)

mapa = folium.Map(location=[-16.6850, -49.2580], zoom_start=13, tiles='CartoDB positron')

cores_folium = {'critico': '#D32F2F', 'alto': '#F57C00', 'moderado': '#FBC02D', 'baixo': '#388E3C'}
raios_folium = {'critico': 10, 'alto': 7, 'moderado': 5, 'baixo': 4}

for _, row in locais.iterrows():
    nivel = row['nivel_contaminacao']
    popup_text = (f"<b>{row['local']}</b><br>"
                  f"Bairro: {row['bairro']}<br>"
                  f"Nivel: {nivel}<br>"
                  f"Atividade: {row['atividade_GBq']} GBq<br>"
                  f"Pessoas afetadas: {row['pessoas_afetadas']}")
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=raios_folium[nivel],
        popup=folium.Popup(popup_text, max_width=240),
        tooltip=row['bairro'],
        color='white', weight=2,
        fill=True, fill_color=cores_folium[nivel], fill_opacity=0.75
    ).add_to(mapa)

heat_data = [[row['lat'], row['lon'], row['atividade_GBq'] / locais['atividade_GBq'].max()]
             for _, row in locais.iterrows()]
HeatMap(heat_data, radius=35, blur=25, min_opacity=0.3).add_to(mapa)

mapa.save("../reports/dashboard/mapa_interativo_goiania.html")
print("Mapa interativo salvo em reports/dashboard/mapa_interativo_goiania.html")


## 4.3 — Correlação Distância × Atividade

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * asin(sqrt(a))

lat0, lon0 = locais.iloc[0]['lat'], locais.iloc[0]['lon']
locais['distancia_km'] = locais.apply(
    lambda r: haversine(lat0, lon0, r['lat'], r['lon']), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
color_map = {'critico': '#D32F2F', 'alto': '#F57C00', 'moderado': '#FBC02D', 'baixo': '#388E3C'}
colors = [color_map[n] for n in locais['nivel_contaminacao']]

axes[0].scatter(locais['distancia_km'], locais['atividade_GBq'],
                s=locais['pessoas_afetadas'] * 3, c=colors, alpha=0.8, edgecolors='white')
axes[0].set_xlabel("Distancia do Epicentro (km)")
axes[0].set_ylabel("Atividade (GBq)")
axes[0].set_title("Atividade vs. Distancia do Epicentro")

axes[1].barh(locais['bairro'], locais['atividade_GBq'], color=colors)
axes[1].set_xlabel("Atividade (GBq)")
axes[1].set_title("Atividade por Ponto de Contaminacao")

plt.tight_layout()
plt.savefig("../reports/figures/04_distancia_atividade.png", dpi=150, bbox_inches='tight')
plt.show()

corr = locais['distancia_km'].corr(locais['atividade_GBq'])
print(f"Correlacao Distancia x Atividade: r = {corr:.3f}")
